# Pset 2 - Preguntas de negocio
## Emilio Soria - 00326990

In [50]:
%pip install psycopg2 pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
import psycopg2
import pandas as pd

In [52]:
conn = psycopg2.connect(
    host="warehouse",
    database="source",
    user="root",
    password="root",
    port=5432
)

In [53]:
def run_query(sql: str) -> pd.DataFrame:
    try:
        return pd.read_sql(sql, conn)
    except Exception as e:
        conn.rollback()
        raise e

## Demanda / Estacionalidad
**1. Viajes por mes (2024)**


### Análisis

En 2024, la cantidad de viajes en taxi muestra variaciones moderadas entre los meses. El mayor número de viajes se registra en octubre (3,828,314 viajes), mientras que el menor ocurre en enero (2,985,433 viajes). En general, se observa una tendencia a un mayor volumen de viajes hacia la segunda mitad del año, lo que sugiere una mayor demanda durante los meses de otoño.

### Tablas usadas
* gold.fct_trips
* gold.dim_date

In [54]:
q1 = """
SELECT
    d.year,
    d.month,
    COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.year, d.month
ORDER BY d.year, d.month;
"""
df1 = run_query(q1)
df1

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,year,month,trips
0,2024,1,2985433
1,2024,2,3024850
2,2024,3,3595437
3,2024,4,3526733
4,2024,5,3735589
5,2024,6,3544924
6,2024,7,3078133
7,2024,8,2977864
8,2024,9,3631133
9,2024,10,3828314


**2. Viajes por service_type y mes**

### Análisis

Los taxis yellow representan la gran mayoría de los viajes en todos los meses de 2024, con más de 2.9 millones de viajes mensuales, mientras que los taxis green registran aproximadamente 50–60 mil viajes por mes. Esto indica que el servicio yellow domina claramente el mercado de transporte en taxi en Nueva York. Además, ambos tipos de servicio muestran un patrón similar de variación mensual, con mayores volúmenes de viajes hacia la segunda mitad del año.

### Tablas usadas

* gold.fct_trips
* gold.dim_date

In [55]:
q2 = """
SELECT
    d.year,
    d.month,
    f.service_type,
    COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.year, d.month, f.service_type
ORDER BY d.year, d.month, f.service_type;
"""
df2 = run_query(q2)
df2

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,year,month,service_type,trips
0,2024,1,green,56370
1,2024,1,yellow,2929063
2,2024,2,green,53403
3,2024,2,yellow,2971447
4,2024,3,green,57254
5,2024,3,yellow,3538183
6,2024,4,green,56257
7,2024,4,yellow,3470476
8,2024,5,green,60781
9,2024,5,yellow,3674808


**3. Top 10 zonas de pickup (total 2024)**

### Análisis

Las zonas con mayor número de recogidas en 2024 se concentran principalmente en Manhattan, lo que refleja su alta densidad de actividad económica y turística. Sin embargo, también destacan JFK Airport y LaGuardia Airport en Queens, lo que indica que los aeropuertos son puntos importantes de origen de viajes en taxi.

### Tablas usadas

* gold.fct_trips
* gold.dim_date
* gold.dim_zone

In [56]:
q3 = """
SELECT
    z.zone AS pickup_zone,
    z.borough AS pickup_borough,
    COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN gold.dim_zone z
    ON f.pu_zone_key = z.zone_key
WHERE d.year = 2024
GROUP BY z.zone, z.borough
ORDER BY trips DESC
LIMIT 10;
"""
df3 = run_query(q3)
df3

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,pickup_zone,pickup_borough,trips
0,JFK Airport,Queens,1910244
1,Upper East Side South,Manhattan,1892772
2,Midtown Center,Manhattan,1886880
3,Upper East Side North,Manhattan,1718376
4,Midtown East,Manhattan,1399892
5,Times Sq/Theatre District,Manhattan,1362178
6,Penn Station/Madison Sq West,Manhattan,1340585
7,Lincoln Square East,Manhattan,1299332
8,LaGuardia Airport,Queens,1275003
9,Murray Hill,Manhattan,1160466


**4. Top 10 zonas de dropoff (total 2024).**

### Análisis

Las zonas con mayor número de dropoffs en 2024 se encuentran exclusivamente en Manhattan, lo que indica que este borough funciona como el principal destino de los viajes en taxi en la ciudad. Esto refleja la importancia de Manhattan como centro económico, turístico y de entretenimiento.

### Tablas usadas

* gold.fct_trips
* gold.dim_date
* gold.dim_zone

In [57]:
q4 = """
SELECT
    z.zone AS dropoff_zone,
    z.borough AS dropoff_borough,
    COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN gold.dim_zone z
    ON f.do_zone_key = z.zone_key
WHERE d.year = 2024
GROUP BY z.zone, z.borough
ORDER BY trips DESC
LIMIT 10;
"""
df4 = run_query(q4)
df4

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,dropoff_zone,dropoff_borough,trips
0,Upper East Side North,Manhattan,1807550
1,Upper East Side South,Manhattan,1714789
2,Midtown Center,Manhattan,1521470
3,Times Sq/Theatre District,Manhattan,1286410
4,Murray Hill,Manhattan,1199480
5,Midtown East,Manhattan,1171128
6,Lincoln Square East,Manhattan,1137196
7,Upper West Side South,Manhattan,1135547
8,East Chelsea,Manhattan,1063507
9,Lenox Hill West,Manhattan,1058186


**5. Top 5 boroughs por mes (pickup).**

### Análisis

Durante todos los meses de 2024, Manhattan concentra la mayor cantidad de viajes iniciados, superando ampliamente a los demás boroughs. Queens ocupa el segundo lugar en la mayoría de los meses. Esto confirma que la mayor actividad de taxis se origina en las zonas máspobladas y con mayor actividad económica de la ciudad.

### Tablas usadas

* gold.fct_trips
* gold.dim_date
* gold.dim_zone

In [58]:
q5 = """
WITH borough_month AS (
    SELECT
        d.year,
        d.month,
        z.borough,
        COUNT(*) AS trips
    FROM gold.fct_trips f
    JOIN gold.dim_date d
        ON f.pickup_date_key = d.date_key
    JOIN gold.dim_zone z
        ON f.pu_zone_key = z.zone_key
    WHERE d.year = 2024
    GROUP BY d.year, d.month, z.borough
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY year, month
            ORDER BY trips DESC
        ) AS rn
    FROM borough_month
)
SELECT
    year,
    month,
    borough,
    trips
FROM ranked
WHERE rn <= 5
ORDER BY year, month, rn
"""
df5 = run_query(q5)
df5

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,year,month,borough,trips
0,2024,1,Manhattan,2651102
1,2024,1,Queens,281604
2,2024,1,Brooklyn,32709
3,2024,1,Unknown,10379
4,2024,1,Bronx,7719
5,2024,2,Manhattan,2712722
6,2024,2,Queens,257742
7,2024,2,Brooklyn,35309
8,2024,2,Unknown,9695
9,2024,2,Bronx,7641


**6. Horas pico (top 5 horas) para cada día de semana**

### Análisis

Los horarios con mayor número de viajes se concentran principalmente entre las 17:00 y las 19:00, lo que coincide con las horas pico de salida del trabajo. Este patrón se repite en varios días de la semana.Esto sugiere que la demanda de taxis aumenta significativamente durante las horas de la tarde y principios de la noche.

### Tablas usadas

* gold.fct_trips

In [59]:
q6 = """
WITH hour_dow AS (
    SELECT
        CASE EXTRACT(DOW FROM f.pickup_ts)::int
            WHEN 0 THEN 'Sunday'
            WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'
            WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'
            WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
        END AS day_of_week,
        EXTRACT(HOUR FROM f.pickup_ts)::int AS pickup_hour,
        COUNT(*) AS trips
    FROM gold.fct_trips f
    WHERE f.pickup_ts IS NOT NULL
    GROUP BY 1, 2
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY day_of_week
            ORDER BY trips DESC
        ) AS rn
    FROM hour_dow
)
SELECT
    day_of_week,
    pickup_hour,
    trips
FROM ranked
WHERE rn <= 5
ORDER BY day_of_week, rn;
"""
df6 = run_query(q6)
df6

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,day_of_week,pickup_hour,trips
0,Friday,18,1792951
1,Friday,17,1663226
2,Friday,19,1653348
3,Friday,15,1500433
4,Friday,16,1491713
5,Monday,18,1502923
6,Monday,17,1474304
7,Monday,15,1349780
8,Monday,16,1340797
9,Monday,14,1291790


**7. Distribución de viajes por día de semana (ranking).**

### Análisis

El día con mayor número de viajes es jueves (4), seguido por miércoles (3) y viernes (5), lo que indica que la demanda de taxis es más alta hacia el final de la semana laboral. En contraste, lunes (1) y domingo (0) presentan el menor número de viajes. Esto sugiere que la actividad de transporte aumenta gradualmente durante la semana y alcanza su punto máximo antes del fin de semana.

### Tablas usadas

* gold.fct_trips

In [60]:
q7 = """
SELECT
    EXTRACT(DOW FROM f.pickup_ts)::int AS day_of_week,
    COUNT(*) AS trips
FROM gold.fct_trips f
WHERE f.pickup_ts IS NOT NULL
GROUP BY 1
ORDER BY trips DESC;
"""
df7 = run_query(q7)
df7

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,day_of_week,trips
0,4,25567561
1,3,24674528
2,5,24621435
3,6,24598201
4,2,23422388
5,0,20873714
6,1,20429796


## Ingreso / Tarifas / propinas

**8. Ingreso total por mes**

### Análisis

Los ingresos mensuales por viajes en taxi durante 2024 muestran variaciones a lo largo del año. El mayor ingreso se registra en octubre, con más de 112 millones de dólares, mientras que los valores más bajos se observan al inicio del año, especialmente en enero y febrero. En general, los ingresos tienden a aumentar hacia la segunda mitad del año, lo que coincide con el incremento en el número de viajes.

### Tablas usadas

* gold.fct_trips
* gold.dim_date

In [61]:
q8 = """
SELECT
    d.year,
    d.month,
    SUM(f.total_amount) AS total_revenue
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.year, d.month
ORDER BY d.year, d.month;
"""
df8 = run_query(q8)
df8

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,year,month,total_revenue
0,2024,1,8.161710e+07
1,2024,2,8.220750e+07
2,2024,3,9.965075e+07
3,2024,4,9.912941e+07
4,2024,5,1.085226e+08
5,2024,6,1.015710e+08
6,2024,7,8.918056e+07
7,2024,8,8.715687e+07
8,2024,9,1.067667e+08
9,2024,10,1.121499e+08


**9. Ingreso total por service_type y mes**

### Análisis

Los ingresos generados por los taxis yellow son significativamente mayores que los de los taxis green durante todos los meses de 2024. Mientras que los taxis yellow generan más de 80–110 millones de dólares mensuales, los taxis green producen alrededor de 1.2–1.5 millones de dólares por mes. Esto confirma que el servicio yellow domina el mercado tanto en número de viajes como en generación de ingresos.

### Tablas usadas

* gold.fct_trips
* gold.dim_date

In [62]:
q9 = """
SELECT
    d.year,
    d.month,
    f.service_type,
    SUM(f.total_amount) AS total_revenue
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.year, d.month, f.service_type
ORDER BY d.year, d.month, f.service_type;
"""
df9 = run_query(q9)
df9

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,year,month,service_type,total_revenue
0,2024,1,green,1.268964e+06
1,2024,1,yellow,8.034814e+07
2,2024,2,green,1.214659e+06
3,2024,2,yellow,8.099284e+07
4,2024,3,green,1.318139e+06
5,2024,3,yellow,9.833261e+07
6,2024,4,green,1.322417e+06
7,2024,4,yellow,9.780699e+07
8,2024,5,green,1.501840e+06
9,2024,5,yellow,1.070207e+08


**10. tip % promedio por mes**

### Análisis

El porcentaje promedio de propina durante 2024 se mantiene relativamente estable, generalmente entre 18% y 23% del valor de la tarifa. El valor más alto se observa en enero, mientras que los valores más bajos se registran en agosto y septiembre. En general, los pasajeros mantienen un nivel de propina bastante consistente a lo largo del año.

### Tablas usadas

* gold.fct_trips
* gold.dim_date

In [63]:
q10 = """
SELECT
    d.year,
    d.month,
    AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.year, d.month
ORDER BY d.year, d.month;
"""
df10 = run_query(q10)
df10

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,year,month,avg_tip_pct
0,2024,1,0.229431
1,2024,2,0.208869
2,2024,3,0.195553
3,2024,4,0.199690
4,2024,5,0.200948
5,2024,6,0.190798
6,2024,7,0.193632
7,2024,8,0.187797
8,2024,9,0.188186
9,2024,10,0.195546


**11. tip % por borough y mes**

### Análisis

El porcentaje promedio de propina varía entre boroughs durante 2024. Manhattan y Queens mantienen valores relativamente estables alrededor del 20%, lo que indica un comportamiento consistente de propinas en estas zonas con alta actividad de taxis.

### Tablas usadas

* gold.fct_trips
* gold.dim_date
* gold.dim_zone

In [64]:
q11 = """
SELECT
    d.year,
    d.month,
    z.borough,
    AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN gold.dim_zone z
    ON f.pu_zone_key = z.zone_key
WHERE d.year = 2024
GROUP BY d.year, d.month, z.borough
ORDER BY d.year, d.month, z.borough;
"""
df11 = run_query(q11)
df11

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,year,month,borough,avg_tip_pct
0,2024,1,Bronx,0.024647
1,2024,1,Brooklyn,0.093868
2,2024,1,EWR,0.477636
3,2024,1,Manhattan,0.216906
4,2024,1,Queens,0.209639
...,...,...,...,...
91,2024,12,Manhattan,0.204486
92,2024,12,Queens,0.160514
93,2024,12,Staten Island,0.247559
94,2024,12,Unknown,0.198576


**12. Top 10 zonas por ingreso total (pickup)**

### Análisis

Las zonas que generan mayores ingresos por viajes en taxi están principalmente asociadas a aeropuertos y áreas centrales de Manhattan. JFK Airport y LaGuardia Airport lideran ampliamente en ingresos totales, lo que refleja la alta demanda de transporte desde y hacia los aeropuertos.

### Tablas usadas

* gold.fct_trips
* gold.dim_zone

In [65]:
q12 = """
SELECT
    z.zone,
    z.borough,
    SUM(f.total_amount) AS total_revenue
FROM gold.fct_trips f
JOIN gold.dim_zone z
    ON f.pu_zone_key = z.zone_key
GROUP BY z.zone, z.borough
ORDER BY total_revenue DESC
LIMIT 10;
"""
df12 = run_query(q12)
df12

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,zone,borough,total_revenue
0,JFK Airport,Queens,5.759422e+08
1,LaGuardia Airport,Queens,3.044757e+08
2,Midtown Center,Manhattan,1.696452e+08
3,Upper East Side South,Manhattan,1.454048e+08
4,Times Sq/Theatre District,Manhattan,1.402372e+08
5,Upper East Side North,Manhattan,1.331540e+08
6,Penn Station/Madison Sq West,Manhattan,1.263885e+08
7,Midtown East,Manhattan,1.243297e+08
8,Lincoln Square East,Manhattan,1.065681e+08
9,Midtown North,Manhattan,1.060630e+08


**13. Top 10 zonas por tip % (pickup) con mínimo N viajes**

### Análisis

Las zonas con mayor porcentaje promedio de propina no necesariamente coinciden con las zonas con mayor número de viajes. Las areas fuera de nueva york y Newark Airport muestran porcentajes de propina inusualmente altos, lo que puede deberse a viajes más largos o tarifas más elevadas.

### Tablas usadas

* gold.fct_trips
* gold.dim_zone

In [66]:
N = 1000

q13 = f"""
SELECT
    z.zone,
    z.borough,
    COUNT(*) AS trips,
    AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_zone z
    ON f.pu_zone_key = z.zone_key
GROUP BY z.zone, z.borough
HAVING COUNT(*) >= {N}
ORDER BY avg_tip_pct DESC
LIMIT 10;
"""
df13 = run_query(q13)
df13

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,zone,borough,trips,avg_tip_pct
0,Outside of NYC,NaN,143808,5.652298
1,Newark Airport,EWR,26003,3.888886
2,Cambria Heights,Queens,9039,1.513687
3,Bay Terrace/Fort Totten,Queens,3376,1.074036
4,Baisley Park,Queens,63214,0.824689
5,Astoria Park,Queens,1287,0.731866
6,Spuyten Duyvil/Kingsbridge,Bronx,14943,0.678676
7,Belmont,Bronx,9540,0.651550
8,East New York,Brooklyn,83737,0.606969
9,Marine Park/Floyd Bennett Field,Brooklyn,2271,0.546195


**14. Comparación cash vs card: viajes, ingreso total, tip %**

### Análisis

Los viajes pagados con tarjeta representan la mayoría de los viajes y generan significativamente más ingresos totales que los pagos en efectivo. Además, el porcentaje promedio de propina es mucho mayor cuando el pago se realiza con tarjeta, mientras que los pagos en efectivo muestran prácticamente 0% de propina registrada. Esto sugiere que las propinas se registran principalmente en pagos electrónicos.

### Tablas usadas

* gold.fct_trips
* gold.dim_payment_type

In [67]:
q14 = """
SELECT
    p.payment_type,
    COUNT(*) AS trips,
    SUM(f.total_amount) AS total_revenue,
    AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_payment_type p
    ON f.payment_type = p.payment_type_key
WHERE p.payment_type IN ('cash', 'card')
GROUP BY p.payment_type
ORDER BY p.payment_type;
"""
df14 = run_query(q14)
df14

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,payment_type,trips,total_revenue,avg_tip_pct
0,card,120613322,3.378653e+09,0.271408
1,cash,24291710,5.549102e+08,0.000020


## Duración / distancia / eficiencia

**15. Duración promedio (min) por mes**

### Análisis

La duración promedio de los viajes en taxi durante 2024 varía aproximadamente entre 15 y 19 minutos. Los viajes más cortos se observan al inicio del año, mientras que hacia septiembre, octubre y diciembre la duración promedio aumenta ligeramente, alcanzando cerca de 19 minutos. Esto podría reflejar factores como mayor congestión del tráfico o viajes más largos en ciertos periodos del año.

### Tablas usadas

* gold.fct_trips
* gold.dim_date

In [68]:
q15 = """
SELECT
    d.year,
    d.month,
    AVG(EXTRACT(EPOCH FROM (f.dropoff_ts - f.pickup_ts)) / 60.0) AS avg_duration_min
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE f.pickup_ts IS NOT NULL
  AND f.dropoff_ts IS NOT NULL
  AND d.year = 2024
GROUP BY d.year, d.month
ORDER BY d.year, d.month;
"""
df15 = run_query(q15)
df15

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,year,month,avg_duration_min
0,2024,1,15.705099
1,2024,2,16.068028
2,2024,3,16.734796
3,2024,4,17.104962
4,2024,5,18.101228
5,2024,6,17.642542
6,2024,7,17.286829
7,2024,8,17.449329
8,2024,9,18.692772
9,2024,10,18.343648


**16. Distancia promedio por mes**

### Análisis

La distancia promedio de los viajes en taxi durante 2024 se mantiene relativamente estable, generalmente entre 4 y 6 millas. Los valores más bajos se observan al inicio del año, mientras que septiembre presenta la mayor distancia promedio, cercana a 6 millas. En general, esto indica que la mayoría de los viajes en taxi en Nueva York corresponden a trayectos relativamente cortos dentro de la ciudad.

### Tablas usadas

* gold.fct_trips
* gold.dim_date

In [69]:
q16 = """
SELECT
    d.year,
    d.month,
    AVG(f.trip_distance) AS avg_trip_distance
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.year, d.month
ORDER BY d.year, d.month;
"""
df16 = run_query(q16)
df16

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,year,month,avg_trip_distance
0,2024,1,4.186944
1,2024,2,4.115018
2,2024,3,4.676122
3,2024,4,5.415792
4,2024,5,5.544076
5,2024,6,5.380544
6,2024,7,5.237329
7,2024,8,5.216554
8,2024,9,5.963979
9,2024,10,5.239993


**17. Velocidad promedio (mph) por borough y franja horaria**

### Análisis

La velocidad promedio de los taxis varía entre boroughs y franjas horarias. Manhattan presenta las velocidades más bajas, lo que refleja la alta congestión del tráfico en esa zona. En contraste, boroughs como Queens, Bronx y Staten Island muestran velocidades promedio más altas, probablemente debido a menor densidad de tráfico y trayectos más largos.

### Tablas usadas

* gold.fct_trips
* gold.dim_zone

In [70]:
q17 = """
WITH base AS (
    SELECT
        z.borough,
        CASE
            WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 6 AND 11 THEN 'morning'
            WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 12 AND 17 THEN 'afternoon'
            WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 18 AND 23 THEN 'evening'
            ELSE 'night'
        END AS time_band,
        f.trip_distance,
        EXTRACT(EPOCH FROM (f.dropoff_ts - f.pickup_ts)) / 3600.0 AS duration_hours
    FROM gold.fct_trips f
    JOIN gold.dim_zone z
        ON f.pu_zone_key = z.zone_key
    WHERE f.pickup_ts IS NOT NULL
      AND f.dropoff_ts IS NOT NULL
      AND f.dropoff_ts > f.pickup_ts
)
SELECT
    borough,
    time_band,
    AVG(trip_distance / NULLIF(duration_hours, 0)) AS avg_mph
FROM base
GROUP BY borough, time_band
ORDER BY borough, time_band;
"""
df17 = run_query(q17)
df17

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,borough,time_band,avg_mph
0,Bronx,afternoon,179.563963
1,Bronx,evening,170.189398
2,Bronx,morning,200.802615
3,Bronx,night,94.154238
4,Brooklyn,afternoon,90.755341
5,Brooklyn,evening,70.725382
6,Brooklyn,morning,145.399195
7,Brooklyn,night,81.168944
8,EWR,afternoon,65.695767
9,EWR,evening,57.983972


**18. Percentiles p50 y p90 de duración por borough**

### Análisis

La duración mediana de los viajes varía entre boroughs. Manhattan presenta viajes más cortos en promedio, con una mediana cercana a 12 minutos, mientras que Queens y Staten Island muestran duraciones más largas. Además, el percentil 90 indica que algunos viajes pueden extenderse considerablemente, especialmente en boroughs con trayectos más largos.

### Tablas usadas

* gold.fct_trips
* gold.dim_zone

In [71]:
q18 = """
WITH durations AS (
    SELECT
        z.borough,
        EXTRACT(EPOCH FROM (f.dropoff_ts - f.pickup_ts)) / 60.0 AS duration_min
    FROM gold.fct_trips f
    JOIN gold.dim_zone z
        ON f.pu_zone_key = z.zone_key
    WHERE f.pickup_ts IS NOT NULL
      AND f.dropoff_ts IS NOT NULL
      AND f.dropoff_ts > f.pickup_ts
)
SELECT
    borough,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY duration_min) AS p50_duration_min,
    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY duration_min) AS p90_duration_min
FROM durations
GROUP BY borough
ORDER BY borough;
"""
df18 = run_query(q18)
df18

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,borough,p50_duration_min,p90_duration_min
0,Bronx,23.166667,61.483333
1,Brooklyn,21.516667,48.550000
2,EWR,0.150000,1.333333
3,Manhattan,11.916667,26.333333
4,Queens,31.933333,60.783333
5,Staten Island,29.050000,72.600000
6,Unknown,14.083333,38.250000
7,NaN,0.766667,60.266667


**19. Top 10 zonas (pickup) por p90 de duración**

### Análisis

Las zonas con mayores duraciones de viaje en el percentil 90 se encuentran principalmente en Staten Island, Queens y Brooklyn. Esto indica que en estas áreas algunos viajes pueden ser significativamente más largos, posiblemente debido a mayores distancias geográficas o menor disponibilidad de rutas directas. En contraste, las zonas más centrales de Manhattan no aparecen entre las de mayor duración.

### Tablas usadas

* gold.fct_trips
* gold.dim_zone

In [72]:
q19 = """
WITH durations AS (
    SELECT
        z.zone,
        z.borough,
        EXTRACT(EPOCH FROM (f.dropoff_ts - f.pickup_ts)) / 60.0 AS duration_min
    FROM gold.fct_trips f
    JOIN gold.dim_zone z
        ON f.pu_zone_key = z.zone_key
    WHERE f.pickup_ts IS NOT NULL
      AND f.dropoff_ts IS NOT NULL
      AND f.dropoff_ts > f.pickup_ts
)
SELECT
    zone,
    borough,
    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY duration_min) AS p90_duration_min
FROM durations
GROUP BY zone, borough
ORDER BY p90_duration_min DESC
LIMIT 10;
"""
df19 = run_query(q19)
df19

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,zone,borough,p90_duration_min
0,Arden Heights,Staten Island,124.040000
1,Freshkills Park,Staten Island,102.086667
2,Far Rockaway,Queens,100.010000
3,Hammels/Arverne,Queens,97.366667
4,Coney Island,Brooklyn,92.671667
5,Eltingville/Annadale/Prince's Bay,Staten Island,90.565000
6,Rockaway Park,Queens,88.193333
7,Brighton Beach,Brooklyn,84.076667
8,Gravesend,Brooklyn,83.196667
9,Cambria Heights,Queens,80.800000


## Rutas / patrones

**20. Top 10 rutas borough→borough por número de viajes**

### Análisis

La mayoría de los viajes ocurren dentro del mismo borough, especialmente en Manhattan, que concentra con diferencia la mayor cantidad de trayectos. También se observan muchos viajes entre Queens y Manhattan, lo que refleja una fuerte conexión de transporte entre estos dos boroughs. En general, Manhattan actúa como el principal origen y destino de viajes en taxi en la ciudad.

### Tablas usadas

* gold.fct_trips
* gold.dim_zone

In [73]:
q20 = """
SELECT
    pu.borough AS pickup_borough,
    dz.borough AS dropoff_borough,
    COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone pu
    ON f.pu_zone_key = pu.zone_key
JOIN gold.dim_zone dz
    ON f.do_zone_key = dz.zone_key
GROUP BY pu.borough, dz.borough
ORDER BY trips DESC
LIMIT 10;
"""
df20 = run_query(q20)
df20

C:\Users\Nicolas Soria\AppData\Local\Temp\ipykernel_45204\1954115920.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,pickup_borough,dropoff_borough,trips
0,Manhattan,Manhattan,133949895
1,Queens,Manhattan,8271569
2,Manhattan,Queens,4706977
3,Queens,Queens,4283620
4,Manhattan,Brooklyn,3369042
5,Queens,Brooklyn,2265663
6,Brooklyn,Brooklyn,1754394
7,Brooklyn,Manhattan,971429
8,Manhattan,Bronx,638247
9,Unknown,Unknown,590812


In [74]:
conn.close()